# Frame potential vs. number of parameters

Loads `results/frame_potential.csv` (written by `frame_potential.save_estimate`,
e.g. via `python run.py frame-potential ...` or `python check.py validate --only ...`)
and plots `F^(t)` against each architecture's parameter count -- the same
comparison the original paper's Fig. 2a makes (and the whole reason that
figure is interesting: parameter count alone is a poor predictor of `F^(t)`).

`n_params` isn't a column in the CSV -- it's derived per row from
`(circuit_num, n_qubits, reps)` via `circuits.n_trainable`, the same function
`experiment_tracker.py` uses for the training side, so the two stay
comparable. A row's `circuit_num` can only be resolved if that number is
still defined in `circuits.py` today -- see the "unresolved circuits" cell
below if some aren't; the row stays in `df` either way, just excluded from
the parameter-count plots specifically.

Repeated measurements of the same `(circuit_num, n_qubits, reps, t)` (e.g.
from re-running a calibration check) are shown twice: once raw (every run,
revealing the actual Monte Carlo spread), once pooled into a single point
per architecture with a proper confidence interval -- using the exact same
`n_pairs`-weighted pooling `frame_potential.Estimate.__add__` does, just
reimplemented here since the CSV stores `frame_potential`/`variance`/`n_pairs`
rather than the raw `total`/`sum_sq` an `Estimate` needs (see that pooling
cell for why this is exact, not an approximation).

In [ ]:
import sys
# repo root, so frame_potential/circuits import the same way the CLI (run.py / check.py) does
sys.path.append("..")

In [ ]:
import math

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
import mplcursors

import frame_potential as fp
import circuits

%matplotlib widget

## Load

In [ ]:
path = "../results/frame_potential.csv"
df = fp.load_frame_potential(path=path)
print(f"{len(df)} rows, {df.groupby(['circuit_num', 'n_qubits', 'reps', 't']).ngroups} unique (circuit_num, n_qubits, reps, t) configs")
df.head()

## n_params per row

`n_trainable` builds and differentiates a probe circuit (see its docstring
in `circuits.py`), so it's computed once per unique `(circuit_num, n_qubits,
reps)` triple and cached, not once per row.

In [ ]:
_n_params_cache = {}
_unresolved = set()

def n_params_for(row):
    key = (int(row.circuit_num), int(row.n_qubits), int(row.reps))
    if key not in _n_params_cache:
        try:
            _n_params_cache[key] = circuits.n_trainable(*key)
        except ValueError:
            _n_params_cache[key] = np.nan
            _unresolved.add(key[0])
    return _n_params_cache[key]

df["n_params"] = df.apply(n_params_for, axis=1)

if _unresolved:
    n_dropped = df["n_params"].isna().sum()
    print(f"circuit number(s) {sorted(_unresolved)} are not defined in the current circuits.py "
          f"(renamed/removed since those rows were saved) -- {n_dropped} row(s) have no n_params "
          "and are excluded from the plots below, but are still in df.")
else:
    print("every row's circuit_num resolved fine.")

## Raw scatter: every run, faceted by `t`

`F^(t)` for different `t` aren't the same quantity (different Haar
reference, `t!`) or scale, so each `t` gets its own panel rather than
sharing one axis. Color is `circuit_num`; hover a point for the exact
`(circuit_num, n_qubits, reps)`.

In [ ]:
plottable = df.dropna(subset=["n_params"]).copy()
circuit_nums = sorted(plottable["circuit_num"].unique())
palette = {c: cm.tab10(i % 10) for i, c in enumerate(circuit_nums)}
t_values = [2]

plt.close("all")
fig, axes = plt.subplots(1, len(t_values), figsize=(5.5 * len(t_values), 4.5), squeeze=False)
axes = axes[0]
scatters = []

for ax, t in zip(axes, t_values):
    sub = plottable[plottable["t"] == t]
    for c in circuit_nums:
        pts = sub[sub["circuit_num"] == c]
        if len(pts):
            s = ax.scatter(pts["n_params"], pts["delta"], color=palette[c], label=f"circuit {c}", alpha=0.8)
            scatters.append((s, pts))
    haar = math.factorial(t)
    ax.axhline(haar, color="grey", linestyle="--", linewidth=1, label=f"Haar (t!={haar})")
    ax.set_yscale("log")
    ax.set_xlabel("number of parameters")
    ax.set_ylabel(f"$F^{{({t})}}$")
    ax.set_title(f"t={t}")
    ax.legend(fontsize=8)

fig.suptitle("Frame potential vs. number of parameters (raw -- every run)")
fig.tight_layout()

cursor = mplcursors.cursor([s for s, _ in scatters], hover=True)
@cursor.connect("add")
def _(sel):
    s = sel.artist
    pts = next(p for sc, p in scatters if sc is s)
    row = pts.iloc[sel.index]
    sel.annotation.set_text(
        f"circuit={int(row.circuit_num)}\n"
        f"n_qubits={int(row.n_qubits)} reps={int(row.reps)}\n"
        f"n_params={int(row.n_params)}\n"
        f"F={row.frame_potential:.4f} (+/-{row.fidelity_error:.4f})"
    )

plt.show()